In [13]:
import os
import re

def clean_text(filename, data_dir="data"):
    input_path = os.path.join(data_dir, filename)
    output_path = os.path.join(data_dir, f"cleaned_{filename}")

    with open(input_path, "r", encoding="utf-8") as file:
        book_text = file.read()

    # 줄바꿈·연속 공백 → 단일 공백으로 치환
    cleaned_text = re.sub(r"\s+", " ", book_text).strip()

    print(f"cleaned_{filename}: {len(cleaned_text):,} characters")

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(cleaned_text)

filenames_list = ["The_Odyssey.txt", "The_Iliad.txt"]
for filename in filenames_list:
    clean_text(filename)

cleaned_The_Odyssey.txt: 696,051 characters
cleaned_The_Iliad.txt: 827,147 characters


In [32]:
import tiktoken # pip install tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "There is nothing dearer to a man than his own country and his parents."

tokens = tokenizer.encode(text)

print("글자수:", len(text), "토큰수", len(tokens))
print(tokens)
print(tokenizer.decode(tokens))
for t in tokens:
    print(f"{t}\t -> {tokenizer.decode([t])}")

글자수: 70 토큰수 16
[1858, 318, 2147, 390, 11258, 284, 257, 582, 621, 465, 898, 1499, 290, 465, 3397, 13]
There is nothing dearer to a man than his own country and his parents.
1858	 -> There
318	 ->  is
2147	 ->  nothing
390	 ->  de
11258	 -> arer
284	 ->  to
257	 ->  a
582	 ->  man
621	 ->  than
465	 ->  his
898	 ->  own
1499	 ->  country
290	 ->  and
465	 ->  his
3397	 ->  parents
13	 -> .


In [3]:
for name in ["cleaned_The_Odyssey.txt", "cleaned_The_Iliad.txt"]:
    with open(name, encoding="utf-8") as f:
        data = f.read()
    print(name, len(data), "chars ->", len(tokenizer.encode(data)), "tokens")

cleaned_The_Odyssey.txt 696052 chars -> 171380 tokens
cleaned_The_Iliad.txt 827148 chars -> 202615 tokens


In [2]:
for name in ["The_Odyssey.txt", "The_Iliad.txt"]:
    with open(name, encoding="utf-8") as f:
        data = f.read()
    print(name, len(data), "chars ->", len(tokenizer.encode(data)), "tokens")

The_Odyssey.txt 698117 chars -> 186230 tokens
The_Iliad.txt 908237 chars -> 282913 tokens


In [41]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, txt, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        # print(token_ids)  # 토큰화 된 전체 텍스트 확인
        print("number of tokens in txt:", len(token_ids))
        
        # Sliding Window를 통한 학습 샘플 생성
        for i in range(0, len(token_ids) - max_length, stride): 
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]  # 입력 대비 +1 토큰 시프트
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# BOM(\ufeff) 자동 제거를 위해 utf-8-sig 인코딩 사용
with open("data/cleaned_The_Odyssey.txt", 'r', encoding='utf-8-sig') as file:
    txt = file.read()

dataset = MyDataset(txt, max_length=32, stride=4)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

number of tokens in txt: 171379


In [40]:
dataiter = iter(train_loader)

x, y = next(dataiter)

print(tokenizer.decode(x[0].tolist()))
print(tokenizer.decode(y[0].tolist()))

 father and son. He, if he so will, shall heal me, which no one else neither god nor man can do.’ “Then I
 and son. He, if he so will, shall heal me, which no one else neither god nor man can do.’ “Then I said
